# TFT training on Colab

Fetches data from this repo, runs Optuna HP search + final fit. Resumable. Approx 30–60 min on T4 GPU.

**Set Runtime → Change runtime type → T4 GPU before running.**

In [ ]:
!pip install -q pytorch-forecasting==1.0.0 lightning==2.1.4 optuna pyarrow

In [ ]:
!wget -q https://raw.githubusercontent.com/SaidkomilMS/dis-tft-data/main/dataset.parquet -O /content/dataset.parquet
!wget -q https://raw.githubusercontent.com/SaidkomilMS/dis-tft-data/main/dataset_meta.json -O /content/dataset_meta.json
!ls -la /content/dataset.parquet /content/dataset_meta.json
import os; os.makedirs('/content/artifacts_tft', exist_ok=True)

In [ ]:
import torch, json, numpy as np, pandas as pd, os
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
df = pd.read_parquet('/content/dataset.parquet')
META = json.loads(open('/content/dataset_meta.json').read())
df['hour_utc'] = pd.to_datetime(df['hour_utc'], utc=True)
df = df.sort_values('hour_utc').reset_index(drop=True)
df['time_idx'] = np.arange(len(df), dtype=np.int64)
df['group'] = 'usd_rub'
print('rows:', len(df), 'horizons:', META['horizons'], 'train_pool_end:', META['splits']['train_pool_end'])

In [ ]:
from pytorch_forecasting import TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import QuantileLoss
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

QUANTILES = [0.1, 0.5, 0.9]
MAX_ENC = 48
MAX_PRED = max(META['horizons'])
TRAIN_CUT = META['splits']['train_pool_end']
STORAGE = 'sqlite:////content/artifacts_tft/optuna_tft.db'
N_TRIALS_TFT = 15

TIME_KNOWN = ['time_idx', 'hour_of_day', 'day_of_week', 'is_peak', 'is_weekend', 'ref_rate',
              'sin_hour', 'cos_hour', 'sin_dow', 'cos_dow', 'hour_of_week', 'sin_hour_of_week', 'cos_hour_of_week',
              'hours_from_peak', 'day_part', 'business_hour']
TIME_UNKNOWN = ['turnover_usd', 'bank_rate', 'spread', 'spread_pct']

def make_ds(target):
    return TimeSeriesDataSet(
        df[df.time_idx <= TRAIN_CUT],
        time_idx='time_idx', target=target, group_ids=['group'],
        max_encoder_length=MAX_ENC, max_prediction_length=MAX_PRED,
        static_categoricals=['group'],
        time_varying_known_reals=TIME_KNOWN,
        time_varying_unknown_reals=TIME_UNKNOWN,
        target_normalizer=GroupNormalizer(groups=['group'], transformation='softplus'),
        add_relative_time_idx=True, add_target_scales=True, add_encoder_length=True,
        allow_missing_timesteps=True,
    )

In [ ]:
def search_hp(target, name, n_trials=N_TRIALS_TFT, max_epochs=12):
    cache = f'/content/artifacts_tft/best_{name}.json'
    if os.path.exists(cache):
        print(f'✓ {name}: best params already saved')
        return json.loads(open(cache).read())
    train_ds = make_ds(target)
    val_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=False, stop_randomization=True)
    train_dl = train_ds.to_dataloader(train=True, batch_size=128, num_workers=2)
    val_dl = val_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    def objective(trial):
        params = {
            'hidden_size': trial.suggest_int('hidden_size', 16, 96, step=16),
            'attention_head_size': trial.suggest_int('attention_head_size', 1, 4),
            'dropout': trial.suggest_float('dropout', 0.05, 0.4),
            'hidden_continuous_size': trial.suggest_int('hidden_continuous_size', 8, 32, step=8),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 5e-3, log=True),
        }
        pl.seed_everything(7)
        model = TemporalFusionTransformer.from_dataset(
            train_ds, **params, loss=QuantileLoss(quantiles=QUANTILES), log_interval=0)
        es = EarlyStopping(monitor='val_loss', patience=3, mode='min')
        trainer = pl.Trainer(max_epochs=max_epochs, accelerator='auto', gradient_clip_val=0.5,
                              callbacks=[es], enable_model_summary=False, enable_progress_bar=False, logger=False)
        trainer.fit(model, train_dl, val_dl)
        return float(trainer.callback_metrics.get('val_loss', float('inf')))
    study = optuna.create_study(study_name=f'tft_{name}', storage=STORAGE, load_if_exists=True,
                                  direction='minimize', sampler=optuna.samplers.TPESampler(seed=7))
    completed = sum(1 for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE)
    remaining = max(0, n_trials - completed)
    print(f'optuna {name!r}: {completed}/{n_trials} done, running {remaining} more')
    if remaining > 0:
        study.optimize(objective, n_trials=remaining,
                       callbacks=[lambda s, t: print(f'  trial {t.number+1} val={t.value:.4f}')])
    best = study.best_params
    open(cache, 'w').write(json.dumps(best, indent=2))
    print(f'best {name}: {best}')
    return best

def train_final(target, name, params, max_epochs=30):
    pred_npy = f'/content/artifacts_tft/preds_{name}.npy'
    idx_npy  = f'/content/artifacts_tft/decoder_idx_{name}.npy'
    if os.path.exists(pred_npy) and os.path.exists(idx_npy):
        print(f'✓ {name}: predictions already saved')
        return np.load(pred_npy), np.load(idx_npy)
    train_ds = make_ds(target)
    val_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=False, stop_randomization=True)
    train_dl = train_ds.to_dataloader(train=True, batch_size=128, num_workers=2)
    val_dl = val_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    pl.seed_everything(7)
    model = TemporalFusionTransformer.from_dataset(
        train_ds, **params, loss=QuantileLoss(quantiles=QUANTILES), log_interval=0)
    es = EarlyStopping(monitor='val_loss', patience=5, mode='min')
    ckpt = ModelCheckpoint(monitor='val_loss', mode='min',
                            dirpath='/content/artifacts_tft', filename=f'tft_{name}')
    trainer = pl.Trainer(max_epochs=max_epochs, accelerator='auto', gradient_clip_val=0.5,
                          callbacks=[es, ckpt], enable_model_summary=False, logger=False)
    trainer.fit(model, train_dl, val_dl)
    full_ds = TimeSeriesDataSet.from_dataset(train_ds, df, predict=True, stop_randomization=True)
    full_dl = full_ds.to_dataloader(train=False, batch_size=256, num_workers=2)
    preds = trainer.predict(model, full_dl, return_predictions=True)
    out = torch.cat(preds, dim=0).cpu().numpy()
    decoder_idx = full_ds.x_to_index(full_ds)['time_idx'].values
    np.save(pred_npy, out)
    np.save(idx_npy, decoder_idx)
    return out, decoder_idx

In [ ]:
best_A = search_hp('turnover_usd', 'A_turnover')
out_A, decoder_idx_A = train_final('turnover_usd', 'A_turnover', best_A)
print('A shape:', out_A.shape)

In [ ]:
best_B = search_hp('bank_rate', 'B_rate')
out_B, decoder_idx_B = train_final('bank_rate', 'B_rate', best_B)
print('B shape:', out_B.shape)

In [ ]:
records = []
for h in META['horizons']:
    cum = out_A[:, :h, :].sum(axis=1)
    records.append(pd.DataFrame({
        'row_idx': decoder_idx_A,
        'p10': cum[:,0], 'p50': cum[:,1], 'p90': cum[:,2],
        'direction': 'A', 'horizon': h,
    }))
for h in META['horizons']:
    records.append(pd.DataFrame({
        'row_idx': decoder_idx_B,
        'p10': out_B[:,0,0], 'p50': out_B[:,0,1], 'p90': out_B[:,0,2],
        'direction': 'B', 'horizon': h,
    }))
final = pd.concat(records, ignore_index=True)
final.to_parquet('/content/predictions_tft.parquet', index=False)
print('wrote /content/predictions_tft.parquet  rows:', len(final))
from google.colab import files
files.download('/content/predictions_tft.parquet')